# CDP 2025 — Mumbai Response Extractor

This notebook reads a CDP Excel file and pulls out **all questions + Mumbai's responses** (disclosure no. `31178`) into a clean Excel sheet.

> **Note:** Mumbai is a *city* — its data lives in the **CDP Cities dataset**, not the States & Regions file.  
> Update `INPUT_FILE` in the Config cell once you have the correct file.

## Step 1 — Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "pandas", "-q"])
print("Ready.")

Ready.


## Step 2 — Configuration

Set the input file and Mumbai's disclosure number here. Change `INPUT_FILE` when you have the CDP Cities dataset.

In [2]:
INPUT_FILE     = "cdp_cities_data/2025_Full_Cities_Public_Data_Separated_by_Question.xlsx"
MUMBAI_DISC_NO = 31178   # cdp_disclosing_org_number for Mumbai
OUTPUT_FILE    = "Mumbai_Responses.xlsx"
SKIP_SHEETS    = {"Introduction", "Summary"}  # non-question sheets to ignore

## Step 3 — Read & Filter

We use `pandas` to read all sheets at once, then keep only rows where the disclosing org is Mumbai.  
Each question sheet has a fixed set of metadata columns (first 13); everything after those is the actual response data.

In [3]:
import pandas as pd
import re

# Metadata columns present in every question sheet — responses start after these
META_COLS = {
    "disclosure_cycle", "cdp_requesting_org_number", "requesting_organization",
    "cdp_disclosing_org_number", "disclosing_organization", "disclosing_org_type",
    "cdp_region", "discloser_country_or_area", "public_status",
    "question_number", "question_text", "row_order", "row_name"
}

print(f"Reading {INPUT_FILE} ...")
all_sheets = pd.read_excel(INPUT_FILE, sheet_name=None)

records = []

for sheet_name, df in all_sheets.items():
    if sheet_name in SKIP_SHEETS or "cdp_disclosing_org_number" not in df.columns:
        continue

    mumbai = df[df["cdp_disclosing_org_number"] == MUMBAI_DISC_NO]
    if mumbai.empty:
        continue

    response_cols = [c for c in df.columns if c not in META_COLS]

    for _, row in mumbai.iterrows():
        q_num  = row.get("question_number", sheet_name)
        q_text = row.get("question_text", "")
        r_name = row.get("row_name", "")

        label = str(q_num) if pd.notna(q_num) else sheet_name
        if pd.notna(r_name) and str(r_name).strip():
            label += f" – {r_name}"
        label_text = str(q_text) if pd.notna(q_text) else ""

        # One row per sub-field — Question / Question Text repeat for each
        for col in response_cols:
            val = row.get(col)
            if pd.notna(val) and str(val).strip():
                # strip "col1_", "col2_" … prefix using "_" as the cut point
                clean_col = re.sub(r"^col\d+_", "", col)

                records.append({
                    "Question":        label,
                    "Question Text":   label_text,
                    "Sub-fields":      clean_col,
                    "Mumbai Response": str(val)
                })

print(f"Found {len(records)} rows for Mumbai (disclosure no. {MUMBAI_DISC_NO}).")
if not records:
    print("\n  Mumbai not found — please update INPUT_FILE to the CDP Cities dataset.")

Reading cdp_cities_data/2025_Full_Cities_Public_Data_Separated_by_Question.xlsx ...


Found 932 rows for Mumbai (disclosure no. 31178).


## Step 4 — Save to Excel

We write the collected records to a styled Excel file with:
- A dark blue header row
- Alternating row shading for readability
- Text wrapping so multi-line responses display correctly

In [4]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

if not records:
    print("Nothing to save — re-run Step 3 after updating INPUT_FILE.")
else:
    df_out = pd.DataFrame(records, columns=["Question", "Question Text", "Sub-fields", "Mumbai Response"])

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Mumbai Responses"

    # --- Header row ---
    ws.append(list(df_out.columns))
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.fill      = PatternFill("solid", fgColor="1F4E79")
        cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 28

    # --- Data rows ---
    alt_fill = PatternFill("solid", fgColor="DCE6F1")
    for i, row_data in enumerate(df_out.itertuples(index=False), start=2):
        ws.append(list(row_data))
        for cell in ws[i]:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            if i % 2 == 0:
                cell.fill = alt_fill

    # --- Column widths ---
    ws.column_dimensions["A"].width = 25   # Question
    ws.column_dimensions["B"].width = 55   # Question Text
    ws.column_dimensions["C"].width = 50   # Sub-fields
    ws.column_dimensions["D"].width = 55   # Mumbai Response

    wb.save(OUTPUT_FILE)
    print(f"Saved {len(records)} rows → '{OUTPUT_FILE}'")

Saved 932 rows → 'Mumbai_Responses.xlsx'
